# VoiceHub: inference, data preparation, and fine-tuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/tts_workflow.ipynb)

This notebook follows one TTS checkpoint through baseline inference, raw-data validation, model-specific dataset creation, training, exact resume, export, and post-training inference.

The concrete example is `nari-labs/Dia-1.6B-0626`. VoiceHub owns Dia's PyTorch graph, tokenizer, DAC target path, training objective, checkpoint loader, and export path; the recipe accepts raw `{text, audio}` records. Other families can require codec codes, mel features, flow targets, or multi-phase batches. Consult the [workflow guides](https://kadirnar.github.io/voicehub/guides/) and [training support matrix](https://kadirnar.github.io/voicehub/models/training-support/) before adapting this notebook.

> Start with all execution flags set to `False`. Registry discovery, manifest loading, validation, and splitting are CPU-safe. Inference downloads a large checkpoint, and full fine-tuning of Dia 1.6B with AdamW optimizer state may exceed the memory available on a free Colab runtime. Use an appropriately sized accelerator, reduce the trainable parameter set where the model recipe supports it, or run the preparation sections separately.

## 0. Install the environment

The default VoiceHub package includes every built-in TTS, ASR, and VAD inference runtime. Add the single `training` extra for this fine-tuning workflow:

```python
%pip install -U "voicehub[training] @ git+https://github.com/kadirnar/voicehub.git@main"
```

For a reproducible run, replace `main` with the release tag or full commit SHA recorded with your model and dataset revisions.

When running from a local VoiceHub clone, set `INSTALL_VOICEHUB=False` in the next cell and use:

```python
%pip install -e ".[training]"
```

Every built-in model graph executes with VoiceHub and PyTorch; no provider framework is installed. Restart the notebook kernel after changing the PyTorch build or VoiceHub revision.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_VOICEHUB = True
# Replace "main" with a release tag or full commit SHA for a recorded run.
VOICEHUB_REVISION = "main"
package = (
    "voicehub[training] @ "
    "git+https://github.com/kadirnar/voicehub.git@"
    f"{VOICEHUB_REVISION}"
)

if INSTALL_VOICEHUB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        package,
    ])
    print("VoiceHub default inference and training environment installed.")

required_modules = (
    "voicehub",
    "torch",
)
missing_modules = [
    name for name in required_modules
    if importlib.util.find_spec(name) is None
]
if missing_modules:
    raise RuntimeError(
        "The Dia training environment is incomplete: "
        + ", ".join(missing_modules)
    )
print("The VoiceHub-native Dia runtime and PyTorch are available.")


In [ ]:
from __future__ import annotations

import gc
import json
import random
from pathlib import Path

from voicehub import (
    AutoInferenceModel,
    AutoModelForTextToSpeech,
    DataCollatorForTTSTraining,
    Trainer,
    TrainingArguments,
    TTSFieldSchema,
    TTSGenerationConfig,
    __version__ as voicehub_version,
)


def preferred_device() -> str:
    try:
        import torch
    except ModuleNotFoundError:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return "mps"
    return "cpu"


def preferred_compute_dtype(device: str) -> str:
    if device != "cuda":
        return "float32"
    import torch

    supports_bf16 = getattr(torch.cuda, "is_bf16_supported", lambda: False)
    return "bfloat16" if supports_bf16() else "float16"


def clear_accelerator_cache() -> None:
    gc.collect()
    try:
        import torch
    except ModuleNotFoundError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mps_backend = getattr(torch.backends, "mps", None)
    mps = getattr(torch, "mps", None)
    if (
        mps_backend is not None
        and mps_backend.is_available()
        and mps is not None
        and hasattr(mps, "empty_cache")
    ):
        mps.empty_cache()


# Expensive or state-changing stages are opt-in.
RUN_INFERENCE = False
RUN_TRAINING = False
RUN_POST_TRAINING_INFERENCE = False
USE_WANDB = False

MODEL_TYPE = "dia"
BASE_MODEL = "nari-labs/Dia-1.6B-0626"
DEVICE = preferred_device()
COMPUTE_DTYPE = preferred_compute_dtype(DEVICE)

DATA_ROOT = Path("data/dia_voice").expanduser()
MANIFEST_PATH = DATA_ROOT / "manifest.jsonl"
OUTPUT_DIR = Path("runs/dia_finetune")
ARTIFACTS_DIR = Path("artifacts")

print({
    "voicehub": voicehub_version,
    "model_type": MODEL_TYPE,
    "base_model": BASE_MODEL,
    "device": DEVICE,
    "compute_dtype": COMPUTE_DTYPE,
})


## 1. Inspect inference and training support

Registry discovery does not import a model's ML runtime or load its weights. Inspect the exact support profile before allocating a checkpoint.

In [ ]:
catalog = {
    model_spec.model_type: model_spec
    for model_spec in AutoInferenceModel.available_models()
}
model_spec = catalog[MODEL_TYPE]
training_spec = model_spec.training

print("Capabilities:", ", ".join(model_spec.capabilities))
print("Inference runtime:", model_spec.install_extra or "default")
print("Training support:", training_spec.support.value)
print("Training family:", training_spec.family_name)
print("Turnkey profile:", training_spec.is_turnkey)
print("Phases:", [phase.name for phase in training_spec.phases])

support_counts = {}
for candidate in catalog.values():
    status = candidate.training.support.value
    support_counts[status] = support_counts.get(status, 0) + 1
print("Registry support counts:", support_counts)

tts_models_by_support = {}
for candidate in catalog.values():
    if candidate.task.value != "text-to-speech":
        continue
    status = candidate.training.support.value
    tts_models_by_support.setdefault(status, []).append(candidate.model_type)
for status, model_types in sorted(tts_models_by_support.items()):
    print(f"TTS {status}:", ", ".join(sorted(model_types)))


## 2. Generate a baseline sample

Keep a baseline generated with a fixed prompt and decoding configuration. Use the same request after fine-tuning so that comparisons are meaningful.

Construction is lazy. Setting `RUN_INFERENCE=True` downloads and loads the checkpoint on the first `generate()` call.

In [ ]:
BASELINE_TEXT = (
    "[S1] VoiceHub keeps inference, data preparation, and training "
    "on one explicit lifecycle."
)
baseline_model = None
baseline_output = None

if RUN_INFERENCE:
    baseline_model = AutoModelForTextToSpeech.from_pretrained(
        BASE_MODEL,
        model_type=MODEL_TYPE,
        backend="native",
        compute_dtype=COMPUTE_DTYPE,
        device=DEVICE,
        lazy_load=True,
    )
    baseline_output = baseline_model.generate(
        BASELINE_TEXT,
        generation_config=TTSGenerationConfig(
            seed=42,
            temperature=1.0,
            max_new_tokens=2048,
            output_file=ARTIFACTS_DIR / "dia_baseline.wav",
        ),
    )
    print(baseline_output.sample_rate, baseline_output.file_path)
    print(baseline_output.metadata)
    try:
        from IPython.display import Audio, display
    except ModuleNotFoundError:
        pass
    else:
        display(Audio(baseline_output.audio, rate=baseline_output.sample_rate))
else:
    print("Baseline inference is disabled; set RUN_INFERENCE=True when ready.")


In [ ]:
# Keep the written sample and release the baseline model before training.
if baseline_output is not None and hasattr(baseline_output.audio, "detach"):
    baseline_output.audio = baseline_output.audio.detach().cpu()
baseline_model = None
clear_accelerator_cache()


## 3. Load an auditable dataset manifest

The Dia dataset consumes `text` and `audio`. Keep stable IDs, speaker/session groups, consent, license, and provenance in the source manifest even when the model adapter does not consume them.

Example JSON Lines record:

```json
{"id":"speaker01-session01-0001","text":"[S1] Exact transcript.","audio":"audio/0001.wav","speaker_id":"speaker01","session_id":"session01","consent":true,"license":"owned"}
```

Only train on voices authorized for the intended use. Keep source recordings immutable and put normalized files in a versioned prepared-data directory.

### Optional: load the manifest from Google Drive

For Colab, a convenient layout is `/content/drive/MyDrive/voicehub-data/dia_voice/manifest.jsonl` with audio below `/content/drive/MyDrive/voicehub-data/dia_voice/audio/`. Keep audio paths inside the manifest relative to the manifest directory. Set the flag below to `True` only in Colab.

In [ ]:
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/voicehub-data/dia_voice")
    MANIFEST_PATH = DATA_ROOT / "manifest.jsonl"

print("Manifest:", MANIFEST_PATH)


In [ ]:
def load_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    records = []
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise TypeError(f"{path}:{line_number} is not a JSON object")
            if "audio" in record:
                audio_path = Path(record["audio"]).expanduser()
                if not audio_path.is_absolute():
                    audio_path = path.parent / audio_path
                record["audio"] = str(audio_path.resolve())
            records.append(record)
    if not records:
        raise ValueError(f"No records found in {path}")
    return records


template_records = [
    {
        "id": "speaker01-session01-0001",
        "text": "[S1] Replace this with an exact transcript.",
        "audio": str(DATA_ROOT / "audio" / "session01-0001.wav"),
        "speaker_id": "speaker01",
        "session_id": "session01",
        "consent": True,
        "license": "owned",
    },
    {
        "id": "speaker01-session02-0001",
        "text": "[S1] Keep validation in a different recording session.",
        "audio": str(DATA_ROOT / "audio" / "session02-0001.wav"),
        "speaker_id": "speaker01",
        "session_id": "session02",
        "consent": True,
        "license": "owned",
    },
]

records = load_jsonl(MANIFEST_PATH) if MANIFEST_PATH.is_file() else template_records
print(f"Loaded {len(records)} record(s)")
if not MANIFEST_PATH.is_file():
    print(f"Template mode: create {MANIFEST_PATH} before training.")


## 4. Normalize and validate Dia audio

This notebook enforces a mono, 44,100 Hz prepared-data policy. VoiceHub's standard-library PCM WAVE decoder and PyTorch resampler perform the file conversion without NumPy, SoundFile, Librosa, or Torchaudio. The Dia adapter rejects wrong sample rates and requires in-memory tensors to be mono rank-1 values.

`prepare_dia_audio()` is an explicit preprocessing helper. Call it when building the prepared dataset, not inside every training epoch. Resampling does not fix clipping, noise, incorrect transcripts, or licensing problems.

In [ ]:
import torch

from voicehub.processing import load_native_audio, load_pcm_wave, save_pcm_wave


def prepare_dia_audio(source: str | Path, destination: str | Path) -> Path:
    source = Path(source)
    destination = Path(destination)
    audio = load_native_audio(
        source,
        target_sampling_rate=44_100,
    )
    return save_pcm_wave(
        destination,
        audio.waveform,
        audio.sampling_rate,
    )


def dia_record_errors(records: list[dict]) -> list[str]:
    errors = []
    seen_ids = set()
    for index, record in enumerate(records):
        record_id = str(record.get("id", index))
        if record_id in seen_ids:
            errors.append(f"{record_id}: duplicate id")
        seen_ids.add(record_id)
        if not str(record.get("text", "")).strip():
            errors.append(f"{record_id}: empty transcript")
        if record.get("consent") is not True:
            errors.append(f"{record_id}: consent is not recorded")
        audio_path = Path(str(record.get("audio", ""))).expanduser()
        if not audio_path.is_file():
            errors.append(f"{record_id}: missing audio {audio_path}")
            continue
        try:
            channels, sample_rate = load_pcm_wave(
                audio_path,
                preserve_channels=True,
            )
        except (OSError, TypeError, ValueError) as error:
            errors.append(f"{record_id}: unreadable audio ({error})")
            continue
        if channels.shape[0] != 1:
            errors.append(
                f"{record_id}: expected mono, got {channels.shape[0]} channels"
            )
        if sample_rate != 44_100:
            errors.append(f"{record_id}: expected 44100 Hz, got {sample_rate} Hz")
        if channels.numel() == 0:
            errors.append(f"{record_id}: empty audio")
        elif not torch.isfinite(channels).all():
            errors.append(f"{record_id}: audio contains non-finite samples")
    return errors


validation_errors = dia_record_errors(records)
if validation_errors:
    print("Dataset is not ready:")
    for error in validation_errors[:20]:
        print(" -", error)
else:
    print("All records satisfy the structural Dia audio contract.")


## 5. Create leakage-resistant splits

Split by speaker or recording session instead of randomly splitting adjacent clips. The helper below keeps every `session_id` in exactly one split. Persist the resulting manifests in a real project.

In [ ]:
def grouped_split(
    records: list[dict],
    *,
    group_key: str = "session_id",
    validation_fraction: float = 0.1,
    seed: int = 42,
) -> tuple[list[dict], list[dict]]:
    if not 0.0 < validation_fraction < 1.0:
        raise ValueError("validation_fraction must be between 0 and 1")
    groups = {}
    for record in records:
        group = record.get(group_key)
        if group is None:
            raise ValueError(f"Every record requires {group_key!r}")
        groups.setdefault(str(group), []).append(record)
    group_names = sorted(groups)
    if len(group_names) < 2:
        raise ValueError(f"At least two {group_key} groups are required")
    random.Random(seed).shuffle(group_names)
    validation_group_count = max(
        1,
        min(len(group_names) - 1, round(len(group_names) * validation_fraction)),
    )
    validation_groups = set(group_names[:validation_group_count])
    train_records = [
        record for record in records
        if str(record[group_key]) not in validation_groups
    ]
    validation_records = [
        record for record in records
        if str(record[group_key]) in validation_groups
    ]
    return train_records, validation_records


train_records, validation_records = grouped_split(
    records,
    group_key="session_id",
    validation_fraction=0.1,
    seed=42,
)
print({"train": len(train_records), "validation": len(validation_records)})


## 6. Build the training runtime and model-specific datasets

Create a fresh lazy wrapper for training. Do not reuse an object already transformed for serving. `validate_training_support()` rejects incompatible checkpoints and backends before allocating weights.

For Dia, `create_training_dataset()` attaches VoiceHub's native processor collator. It creates text inputs, delayed decoder inputs, masks, and DAC codec labels from raw records.

In [ ]:
training_model = None
train_dataset = None
validation_dataset = None

if RUN_TRAINING:
    if validation_errors:
        raise RuntimeError("Fix the dataset validation errors before training.")
    training_model = AutoModelForTextToSpeech.from_pretrained(
        BASE_MODEL,
        model_type=MODEL_TYPE,
        backend="native",
        compute_dtype=COMPUTE_DTYPE,
        device=DEVICE,
        lazy_load=True,
    )
    selected_spec = training_model.validate_training_support()
    print(selected_spec.support.value, selected_spec.family_name)
    train_dataset = training_model.create_training_dataset(train_records)
    validation_dataset = training_model.create_training_dataset(validation_records)
    print(len(train_dataset), len(validation_dataset))
else:
    print("Dataset construction is disabled; set RUN_TRAINING=True when ready.")


In [ ]:
def describe_batch(batch: dict) -> dict:
    description = {}
    for name, value in batch.items():
        shape = getattr(value, "shape", None)
        description[name] = tuple(shape) if shape is not None else type(value).__name__
    return description


if train_dataset is not None:
    preview_count = min(2, len(train_dataset))
    features = [train_dataset[index] for index in range(preview_count)]
    preview_batch = train_dataset.collate_fn(features)
    print(describe_batch(preview_batch))
else:
    print("No training batch to inspect in smoke mode.")


### When a model requires preprocessed tensors

The Dia path above is raw-data capable. Preprocessed profiles such as Vui, Kokoro, Echo, F5-TTS, GPT-SoVITS, MeloTTS, OuteTTS, StyleTTS2, Qwen3-TTS, XTTS, Bark, and VITS require model-shaped inputs. `DataCollatorForTTSTraining` performs structural padding but does not invent codec delays, flow noise, velocity targets, alignments, or GAN pairs.

Declare ambiguous sequence axes explicitly:

In [ ]:
mel_collator = DataCollatorForTTSTraining(
    field_schemas={
        "model_inputs.mel": TTSFieldSchema(
            sequence_dim=-1,
            padding_side="right",
            length_field="mel_lengths",
            mask_field="mel_mask",
            pad_to_multiple_of=8,
        ),
    },
)
print(mel_collator.field_schemas)


## 7. Configure a one-step training smoke test

Keep `MAX_STEPS=1` until one batch produces a finite differentiable loss, intended parameters receive gradients, frozen codec parameters stay frozen, and saving/reloading succeeds. Increase it only after the smoke test.

The built-in strategy is single-process PyTorch. Exact generic mid-epoch resume requires a stable sized dataset and `dataloader_num_workers=0`.

Set `USE_WANDB=True` to enable the W&B callback installed by `voicehub[training]`. Authenticate with `wandb login` or `WANDB_API_KEY`; never put credentials in notebook cells or serialized training arguments.

In [ ]:
MAX_STEPS = 1
RESUME_FROM_CHECKPOINT: bool | str = False
trainer = None

if RUN_TRAINING:
    arguments = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        max_steps=MAX_STEPS,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=5e-5,
        warmup_ratio=0.03,
        max_grad_norm=1.0,
        bf16=COMPUTE_DTYPE == "bfloat16",
        fp16=COMPUTE_DTYPE == "float16",
        use_cpu=DEVICE == "cpu",
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=1,
        save_strategy="steps",
        save_steps=1,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        dataloader_num_workers=0,
        seed=42,
        data_seed=42,
        report_to="wandb" if USE_WANDB else "none",
        wandb_project="voicehub-finetuning",
        wandb_tags=["tts", "dia", "fine-tuning"],
        wandb_log_model="end" if USE_WANDB else False,
    )
    trainer = Trainer(
        model=training_model,
        args=arguments,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
    )
    print(arguments)
else:
    print("Trainer construction is disabled in smoke mode.")


In [ ]:
train_output = None
if trainer is not None:
    # False starts only in a new or checkpoint-free output directory.
    # True resumes the newest complete checkpoint.
    # A string selects one explicit checkpoint directory.
    train_output = trainer.train(
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    )
    print(train_output)
else:
    print("Training is disabled in smoke mode.")


## 8. Save, resume, and export

`checkpoint-N/` is an exact-resume artifact containing model, optimizer, scheduler, random state, callbacks, sampler, strategy, and recipe topology. `trainer.save_model()` writes a portable VoiceHub artifact and, where supported, a namespaced `native_export/`.

A safetensors file is a weight container, not an exact resume. Do not treat GGUF, ONNX, TensorRT, JIT, vLLM, or a quantized serving artifact as a generic trainable checkpoint.

In [ ]:
final_artifact = OUTPUT_DIR / "final"
if trainer is not None and train_output is not None:
    final_artifact = trainer.save_model(final_artifact)
    print("Saved:", final_artifact)
else:
    print("No artifact was written in smoke mode.")


In [ ]:
# Keep `trainer` alive instead if you plan to evaluate or resume in this kernel.
training_model = None
train_dataset = None
validation_dataset = None
trainer = None
clear_accelerator_cache()


## 9. Reload the fine-tuned artifact for inference

Load the root portable artifact through VoiceHub. Use `native_export/` only when the model recipe documents it as a complete source-native inference export.

Compare the baseline and fine-tuned samples with the same prompt, seed, and decoding settings. Human listening tests should cover intelligibility, prosody, artifacts, speaker similarity where applicable, memorization, and safety. On a constrained accelerator, saving the artifact and reloading it in a fresh kernel is safer than retaining multiple model runtimes.

In [ ]:
fine_tuned_output = None
if RUN_POST_TRAINING_INFERENCE:
    if not final_artifact.is_dir():
        raise FileNotFoundError(f"Fine-tuned artifact not found: {final_artifact}")
    fine_tuned_model = AutoModelForTextToSpeech.from_pretrained(
        final_artifact,
        device=DEVICE,
        lazy_load=True,
    )
    fine_tuned_output = fine_tuned_model.generate(
        BASELINE_TEXT,
        generation_config=TTSGenerationConfig(
            seed=42,
            temperature=1.0,
            max_new_tokens=2048,
            output_file=ARTIFACTS_DIR / "dia_finetuned.wav",
        ),
    )
    print(fine_tuned_output.sample_rate, fine_tuned_output.file_path)
    try:
        from IPython.display import Audio, display
    except ModuleNotFoundError:
        pass
    else:
        display(Audio(fine_tuned_output.audio, rate=fine_tuned_output.sample_rate))
else:
    print("Post-training inference is disabled.")


## 10. Adapting the notebook to another family

The discovery cell prints the live TTS membership of every support level, so use it instead of copying a stale model list. The support enum describes execution readiness; it does not replace the model-specific data contract.

| Registry support | Contract |
| --- | --- |
| `native` | VoiceHub can reach an integrated differentiable objective. Read the matrix to learn whether the selected checkpoint accepts raw records, pre-encoded targets, or an explicit acoustic configuration. |
| `preprocessed` | The recipe is executable, but callers must supply model-shaped tokens, features, alignments, spectrograms, codec IDs, or adversarial pairs exactly as documented. |
| `custom` | A built-in specialized adapter owns model-specific phases or safety gates; it is not interchangeable with a generic Trainer objective. |
| `inference-only` | No trainable graph exists. The current fixed WebRTC and Auditok VAD algorithms are the only entries in this category; no TTS integration is silently assigned a generic loss. |

Current raw-record examples include Dia, ConversationTTS, LLaSA, Parler-TTS, Irodori-TTS, VoxCPM, OmniVoice, Higgs Audio, CSM, NeuTTS, and SpeechT5. Custom profiles are Chatterbox, CosyVoice, and OpenVoice. Before changing `MODEL_TYPE`, read the [training support matrix](https://kadirnar.github.io/voicehub/models/training-support/) for the exact checkpoint, objective, phase, frozen-component, and export boundary.

### Final checklist

- The selected checkpoint is the differentiable training variant.
- Dataset consent, provenance, license, and split policy are recorded.
- Audio and transcripts satisfy the model-specific contract.
- One batch produces a finite scalar loss with gradients.
- Frozen codecs/vocoders receive no gradients.
- A one-step checkpoint resumes without changing the training plan.
- The final portable artifact reloads for inference.
- Baseline and fine-tuned evaluation use identical generation settings.
